# Lec 03 — Cost 함수 최소화 (Gradient Descent)

Lec02에서 본 학습 루프의 **이론적 배경**을 직접 시각화로 확인합니다.

다루는 내용:
1. **단순화한 가설** $H(x) = Wx$ (b를 빼서 1차원 풍경으로)
2. Cost 함수의 **모양 그리기** — convex(볼록) 인지 직접 확인
3. **미분 식을 손으로 유도**하고 코드로 구현
4. Gradient Descent로 W가 최소점으로 굴러가는 과정 시각화
5. **Learning rate** 를 바꿔서 발산/수렴 직접 보기
6. `tf.keras.optimizers.SGD` 와 결과 비교

## 0. 환경 셋업

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

tf.random.set_seed(0)
np.random.seed(0)

print("TF:", tf.__version__)

OUT = "../../outputs/season1/lec03"
os.makedirs(OUT, exist_ok=True)

## 1. 데이터 & 단순화한 가설

$$
H(x) = W \cdot x \qquad (\text{bias } b\text{는 0으로 고정})
$$

이렇게 하면 cost가 **W 하나만의 함수**가 되어 풍경을 1차원으로 그릴 수 있어요.

In [ ]:
x_data = tf.constant([1.0, 2.0, 3.0])
y_data = tf.constant([1.0, 2.0, 3.0])

def cost_fn(W):
    """H(x) = Wx 일 때의 MSE cost."""
    h = W * x_data
    return tf.reduce_mean((h - y_data) ** 2)

print("cost(W=0)  =", cost_fn(0.0).numpy())   # (0-1)^2+(0-2)^2+(0-3)^2)/3 = 14/3
print("cost(W=1)  =", cost_fn(1.0).numpy())   # 0 ← 최소
print("cost(W=2)  =", cost_fn(2.0).numpy())
print("cost(W=-1) =", cost_fn(-1.0).numpy())

## 2. Cost 풍경 직접 그려보기

W를 -3부터 5까지 훑으면서 cost를 계산 → 그래프로. **그릇 모양(convex)** 이 나오면 GD가 안전하게 동작.

In [ ]:
Ws = np.linspace(-3, 5, 200)
costs = [cost_fn(w).numpy() for w in Ws]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(Ws, costs)
ax.axvline(1.0, color="red", linestyle="--", alpha=0.5, label="true minimum (W=1)")
ax.set_xlabel("W")
ax.set_ylabel("cost(W)")
ax.set_title("Cost as a function of W  —  convex bowl")
ax.legend(); ax.grid(True, alpha=0.3)
fig.savefig(f"{OUT}/cost_shape.png", dpi=120, bbox_inches="tight")
plt.show()

## 3. 미분 식 — 손으로 유도

$$
\text{cost}(W) = \frac{1}{m}\sum_{i=1}^{m} (Wx_i - y_i)^2
$$

$$
\frac{\partial \text{cost}}{\partial W}
= \frac{2}{m}\sum_{i=1}^{m} (Wx_i - y_i)\cdot x_i
$$

이 식을 코드로 옮긴 게 `manual_grad`.
비교: `tf.GradientTape` 가 자동으로 같은 값을 뽑아주는지 확인.

In [ ]:
def manual_grad(W):
    h = W * x_data
    return tf.reduce_mean(2 * (h - y_data) * x_data)

def auto_grad(W_var):
    with tf.GradientTape() as tape:
        c = cost_fn(W_var)
    return tape.gradient(c, W_var)

for w in [-1.0, 0.0, 1.0, 2.0, 3.0]:
    Wv = tf.Variable(w)
    print(f"W={w:+.1f}  manual={manual_grad(w).numpy():+.4f}   auto={auto_grad(Wv).numpy():+.4f}")

## 4. Gradient Descent — W가 굴러내려가는 과정

`W = -3` 같은 멀리서 출발해서 한 스텝씩 어떻게 이동하는지 풍경 위에 점으로 찍기.

In [ ]:
def run_gd(w_init=-3.0, lr=0.1, steps=30):
    W = tf.Variable(w_init)
    trace = [W.numpy()]
    costs = [cost_fn(W).numpy()]
    for _ in range(steps):
        with tf.GradientTape() as tape:
            c = cost_fn(W)
        dW = tape.gradient(c, W)
        W.assign_sub(lr * dW)
        trace.append(W.numpy())
        costs.append(cost_fn(W).numpy())
    return np.array(trace), np.array(costs)

trace, costs_along = run_gd(w_init=-3.0, lr=0.1, steps=30)
print(f"start W={trace[0]:.3f}  →  end W={trace[-1]:.4f}  (target=1.0)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(Ws, costs, color="lightgray", label="cost(W)")
axes[0].scatter(trace, costs_along, c=range(len(trace)), cmap="viridis", s=30, zorder=3)
axes[0].plot(trace, costs_along, color="black", alpha=0.3, linewidth=1)
axes[0].axvline(1.0, color="red", linestyle="--", alpha=0.5)
axes[0].set_xlabel("W"); axes[0].set_ylabel("cost")
axes[0].set_title("GD on the cost landscape")
axes[0].grid(True, alpha=0.3); axes[0].legend()

axes[1].plot(costs_along, marker="o")
axes[1].set_xlabel("step"); axes[1].set_ylabel("cost")
axes[1].set_title("Cost vs step")
axes[1].grid(True, alpha=0.3)

fig.savefig(f"{OUT}/gd_descent.png", dpi=120, bbox_inches="tight")
plt.show()

## 5. Learning rate 실험 — 수렴 / 진동 / 발산

lr만 바꿔서 같은 풍경을 굴려보면 lr이 학습의 운명을 어떻게 좌우하는지 한눈에.

In [ ]:
lrs = [0.01, 0.1, 0.5, 1.0]
results = {lr: run_gd(w_init=-3.0, lr=lr, steps=30) for lr in lrs}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for lr, (tr, cs) in results.items():
    axes[0].plot(Ws, costs, color="lightgray", alpha=0.4)
    axes[0].plot(tr, cs, marker="o", markersize=3, label=f"lr={lr}")
    axes[1].plot(cs, marker="o", markersize=3, label=f"lr={lr}")

axes[0].set_title("Trajectories on the landscape")
axes[0].set_xlabel("W"); axes[0].set_ylabel("cost")
axes[0].set_xlim(-4, 5); axes[0].set_ylim(-1, 30)
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].set_title("Cost vs step")
axes[1].set_xlabel("step"); axes[1].set_ylabel("cost")
axes[1].set_yscale("log")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

fig.savefig(f"{OUT}/lr_compare.png", dpi=120, bbox_inches="tight")
plt.show()

print("\n최종 W 값들:")
for lr, (tr, _) in results.items():
    print(f"  lr={lr:<5}  final W = {tr[-1]:+.4f}")

**해석:**
- `lr=0.01` — 너무 작아서 30 step 안에 못 도착
- `lr=0.1`  — 부드럽게 수렴 ✅
- `lr=0.5`  — 빠르지만 진동하면서 수렴
- `lr=1.0`  — 그릇의 양 벽을 튕기며 **발산** (이 문제에선 lr=1이 임계점 근처)

lr=1을 더 키워서 `lr=1.5` 로 돌려보세요 — cost가 천문학적으로 커질 거예요.

## 6. `tf.keras.optimizers.SGD` 로 같은 일 하기

직접 `assign_sub` 하는 자리에 옵티마이저 객체를 끼워넣을 수 있어요. 결과는 동일.

In [ ]:
W = tf.Variable(-3.0)
opt = tf.keras.optimizers.SGD(learning_rate=0.1)

for step in range(31):
    with tf.GradientTape() as tape:
        c = cost_fn(W)
    grads = tape.gradient(c, [W])
    opt.apply_gradients(zip(grads, [W]))   # ← 이 한 줄이 우리의 W.assign_sub(lr*dW)
    if step % 5 == 0:
        print(f"step={step:3d}  W={W.numpy():+.4f}  cost={c.numpy():.6f}")

## 7. 정리

- Cost 풍경이 **convex** 이면 GD는 어디서 출발해도 global minimum에 도달
- **미분 = 풍경의 기울기**, GD는 "기울기 반대"로 한 발씩
- **lr 가 너무 크면** → 그릇 벽을 튕기며 발산
- **lr 가 너무 작으면** → 수렴은 안전하지만 너무 느림
- `optimizer.apply_gradients(...)` ≡ `W.assign_sub(lr*dW)` (옵티마이저는 이걸 일반화한 도구)

다음 (Lec04) 부터 입력 변수가 여러 개로 늘어나면, 풍경이 1차원 → 다차원이 되면서 시각화는 어려워지지만 **알고리즘은 똑같이 동작**합니다.

## 마무리 — 체크리스트

- [ ] cost(W) 곡선이 그릇 모양임을 직접 그려서 확인
- [ ] 손으로 유도한 미분과 `tape.gradient` 결과가 일치하는 것 확인
- [ ] `lr=0.1` 로 W가 1.0 으로 수렴하는 궤적 시각화
- [ ] `lr` 를 0.01 / 0.1 / 0.5 / 1.0 / 1.5 로 바꿔가며 거동 비교
- [ ] `tf.keras.optimizers.SGD` 로 동일 결과 재현